In [1]:
import subprocess
import pandas as pd
import os, csv
import spacy, spacy_transformers
from string import punctuation
from wordfreq import zipf_frequency

C:\Users\dhima\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# def count_csv_elements_in_file(filepath):
#     total_elements=0
#     with open(filepath, 'r', encoding="latin-1") as file:
#         csv_reader=csv.reader(file)
#         for row in csv_reader:
#             total_elements+=len(row)
#     return total_elements
# language=[]
# total_words=[]
#
# for  path, subdir, files in os.walk('raw-word-lists'):
#     for name in files:
#         filepath = (os.path.join(path, name))
#         language+=[name.split('.')[0]]
#         total_words+=[count_csv_elements_in_file(filepath)]


In [2]:
# pd.DataFrame({'language':language,'total_words':total_words}).to_csv('clean-word-lists.csv',index=False)
df=pd.read_csv('clean-word-lists.csv')
df.drop(df.index[:28], inplace=True)
df.reset_index(inplace=True)
df.drop(columns=['index'], inplace=True)
df

,language,total_words
0,Catalan,3585
1,Croatian,3766
2,Danish,1800765
3,Dutch,173556
4,English,466434
5,Finnish,91672
6,French,336528
7,German,1707903
8,Greek,35279
9,Italian,661563


In [4]:
spacy_models={
    'Catalan':"ca_core_news_trf",
    'Croatian':"hr_core_news_lg",
    'Danish':"da_core_news_trf",
    'Dutch':"nl_core_news_lg",
    'English':"en_core_web_trf",
    'Finnish':"fi_core_news_lg",
    'French':"fr_dep_news_trf",
    'German':"de_dep_news_trf",
    'Greek':"el_core_news_lg",
    'Italian':"it_core_news_lg",
     'Polish':"pl_core_news_lg",
    'Portuguese':"pt_core_news_lg",
    'Romanian':"ro_core_news_lg",
    'Russian':"ru_core_news_lg",
    'Slovenian':"sl_core_news_trf",
    'Spanish':"es_dep_news_trf",
    'Swedish':"sv_core_news_lg",
'Ukranian':"uk_core_news_trf"
}
for lang, model in spacy_models.items():
    # Check if model is already installed
    try:
        spacy.load(model)
        print(f"✅ {lang} ({model}) - Already installed, skipping...")
        continue
    except OSError:
        print(f"⬇️  {lang} ({model}) - Not found, downloading...")

    # Download the model
    result = subprocess.run(
        ['python', '-m', 'spacy', 'download', model],
        capture_output=True,
        text=True
    )

    print(result.stdout)
    if result.stderr:
        print(result.stderr)

    # Check if download was successful
    if result.returncode == 0:
        print(f"✅ Successfully downloaded {model}\n")
    else:
        print(f"❌ Failed to download {model}\n")


In [5]:
# making new dir for clean words
try:
    os.mkdir('data')
    for language in spacy_models.keys():
        try:
            os.mkdir(f'data/{language}')
            print(f"Directory {language} created")
        except FileExistsError:
            print(f'Directory {language} exits')
except FileExistsError:
    print(f'DATA dir  exits')



DATA dir  exits


defining sub functions

In [6]:
def loan_and_clean_word_list(language:str) -> pd.DataFrame:
    with open(f'raw-word-lists/{language}/{language}.txt', 'r', encoding='latin-1') as f:
        word_list = f.read().split(',')
        word_df= pd.DataFrame({
        'word':word_list
    })
    word_df['word']= word_df['word'].str.strip(punctuation)
    return word_df

In [7]:
# nlp=spacy.load(spacy_models[language], disable=['parser', 'tagger', 'ner'])
# getting the base word for the lang
def add_lemma(
        df: pd.DataFrame,
        nlp,
        batch_size:int=2000 ) -> pd.DataFrame:
    docs= nlp.pipe(df['word'].tolist(), batch_size=batch_size )
    lemmas = [doc[0].lemma_ for doc in docs]
    df['lemma'] = pd.DataFrame(lemmas, index=df.index)
    return df

# getting how often word is being used i.e lemma
def word_frequency(
        df: pd.DataFrame,
        language:str
)->pd.DataFrame:
    language_group= spacy_models[language].split('_')[0]
    df['zipf_freq_lemma']=[zipf_frequency(word, lang=language_group) for word in df['lemma']]
    return df

def clean_up_and_export(
        df: pd.DataFrame,
        language:str
)-> None:
    os.makedirs(f'data/{language}', exist_ok=True)  # ONLY CHANGE - creates directory
    df=(
        df.loc[df.groupby('lemma', sort=False)['zipf_freq_lemma'].idxmax()].reset_index(drop=True)
    )
    df=df[df['zipf_freq_lemma']>0]
    df.loc[:, 'word_difficulty']=pd.cut(
        df['zipf_freq_lemma'], bins=[-float('inf'),2.0,4.0 ,float('inf')],
        labels=['advanced', 'intermediate', 'beginnner'],include_lowest=True, right=True
    )
    df.drop(columns=['zipf_freq_lemma','word'], inplace=True)
    df.rename(columns={'lemma':'word'})
    df.to_json(f'data/{language}/word-list-cleaned',orient='index')

In [8]:
# def
# language_group= spacy_models[language].split('_',2)[0]
# language_group

In [9]:
def create_clean_word_list(language:str) -> None:
    nlp=spacy.load(spacy_models[language], disable=['parser', 'textcar', 'ner'])
    print('load in data')
    lang_df=loan_and_clean_word_list(language)
    print('lemmatize word')
    lang_df= add_lemma(lang_df, nlp)
    print('adding zipf freq')
    lang_df=word_frequency(lang_df, language)
    print('do the final clean ups and export')
    clean_up_and_export(lang_df, language)
    return None


In [10]:
# Except Spanish , German and English we are not going to use other lang due to:
#     1. lemmitation runtime is more than 2 hour for each language
#     2. I have a friend that know Spanish and German to verify but not for others
#
# # create_clean_word_list('Catalan')
# create_clean_word_list('Spanish')
# # create_clean_word_list('Danish')
# create_clean_word_list('German')
# # create_clean_word_list('Portuguese')
# # create_clean_word_list('Italian')
# create_clean_word_list('English')
# # create_clean_word_list('Swedish')
# # create_clean_word_list('Dutch')
# # create_clean_word_list('Finnish')
# # create_clean_word_list('French')
# # create_clean_word_list('Greek')
# # create_clean_word_list('Polish')
# # create_clean_word_list('Romanian')
# # create_clean_word_list('Russian')
# # create_clean_word_list('Slovenian')
# # create_clean_word_list('Ukranian')


load in data
lemmatize word
adding zipf freq
do the final clean ups and export
load in data
lemmatize word
adding zipf freq
do the final clean ups and export
load in data
lemmatize word


KeyboardInterrupt: 

In [3]:
!pip uninstall -y ca_core_news_trf hr_core_news_lg da_core_news_trf nl_core_news_lg en_core_web_trf fi_core_news_lg fr_dep_news_trf de_dep_news_trf el_core_news_lg it_core_news_lg pl_core_news_lg pt_core_news_lg ro_core_news_lg ru_core_news_lg sl_core_news_trf es_dep_news_trf sv_core_news_lg uk_core_news_trf

Found existing installation: ca_core_news_trf 3.8.0
Uninstalling ca_core_news_trf-3.8.0:
  Successfully uninstalled ca_core_news_trf-3.8.0
Found existing installation: hr_core_news_lg 3.8.0
Uninstalling hr_core_news_lg-3.8.0:
  Successfully uninstalled hr_core_news_lg-3.8.0
Found existing installation: da_core_news_trf 3.8.0
Uninstalling da_core_news_trf-3.8.0:
  Successfully uninstalled da_core_news_trf-3.8.0
Found existing installation: nl_core_news_lg 3.8.0
Uninstalling nl_core_news_lg-3.8.0:
  Successfully uninstalled nl_core_news_lg-3.8.0
Found existing installation: en_core_web_trf 3.8.0
Uninstalling en_core_web_trf-3.8.0:
  Successfully uninstalled en_core_web_trf-3.8.0
Found existing installation: fi_core_news_lg 3.8.0
Uninstalling fi_core_news_lg-3.8.0:
  Successfully uninstalled fi_core_news_lg-3.8.0
Found existing installation: fr_dep_news_trf 3.8.0
Uninstalling fr_dep_news_trf-3.8.0:
  Successfully uninstalled fr_dep_news_trf-3.8.0
Found existing installation: de_dep_news_t